# Notebook 04 — Food Security Classification

**Objective:** Predict IPC food security phase (1/2/3) per county using NASA weather features.

**This is the core ML module — directly automates what FEWS NET analysts do manually.**

**Models:** Rule-based baseline → Logistic Regression → XGBoost (primary)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.models import (
    prepare_classification_data,
    train_baseline_classifier, train_xgboost_classifier,
    evaluate_classifier, plot_confusion_matrix,
    compare_models, save_model
)
from src.utils import section
import shap

master = pd.read_csv("data/processed/master_dataset.csv")
print("Master shape:", master.shape)
print("IPC phase distribution:")
print(master["ipc_phase_county"].value_counts().sort_index())

## Step 1 — Baseline Model

In [ ]:
section("BASELINE — Rule-Based Classifier")

# Rule: if SPI-3 < -1 → Phase 3 (Crisis), if SPI-3 < 0 → Phase 2, else Phase 1
X_train, X_test, y_train, y_test, features = prepare_classification_data(master)

def rule_based_predict(X):
    preds = []
    for _, row in X.iterrows():
        spi = row.get("spi_3", 0) or 0
        if spi < -1.0:
            preds.append(3)
        elif spi < 0:
            preds.append(2)
        else:
            preds.append(1)
    return preds

y_rule_pred = rule_based_predict(X_test)
from sklearn.metrics import f1_score, classification_report
baseline_f1 = f1_score(y_test, y_rule_pred, average="weighted")
print(f"Baseline Rule F1: {baseline_f1:.4f}")
print(classification_report(y_test, y_rule_pred))

## Step 2 — Logistic Regression

In [ ]:
section("MODEL 2 — Logistic Regression Baseline")
lr = train_baseline_classifier(X_train, y_train)
lr_results = evaluate_classifier(lr, X_test, y_test, "Logistic Regression")
plot_confusion_matrix(lr, X_test, y_test, "Logistic Regression")

## Step 3 — XGBoost Classifier (Primary Model)

In [ ]:
section("MODEL 3 — XGBoost Classifier (Primary)")
xgb_model = train_xgboost_classifier(X_train, y_train)
le = xgb_model.label_encoder_
xgb_results = evaluate_classifier(
    xgb_model, X_test, y_test, "XGBoost", label_encoder=le
)
plot_confusion_matrix(xgb_model, X_test, y_test, "XGBoost", label_encoder=le)

## Step 4 — SHAP Feature Importance

In [ ]:
section("SHAP — Feature Importance")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=features,
                  class_names=["Phase 1", "Phase 2", "Phase 3"],
                  show=False)
plt.title("SHAP Feature Importance — IPC Phase Prediction")
plt.tight_layout()
plt.savefig("reports/figures/07_shap_ipc.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 5 — Model Comparison & Save Best

In [ ]:
section("COMPARISON — All Models")
results = [
    {"model": "Rule-Based", "f1_weighted": baseline_f1, "accuracy": baseline_f1},
    lr_results,
    xgb_results,
]
comparison = compare_models(results)

# Save the best model
save_model(xgb_model, "xgboost_ipc_classifier")
print("\nBest model saved: models/saved/xgboost_ipc_classifier.pkl")